# Data Quality Check

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("Hotel_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset at {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

# Auto-parse columns that look like dates
date_cols = [col for col in df.columns if "date" in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head()


Rows: 5,000 | Columns: 12


,booking_id,booking_date,check_in_date,channel_id,rate_code_id,rooms_sold,gross_room_revenue,status,is_commissionable,default_commission_rate,commission_amount,net_room_revenue
0,BK_00000,2026-10-17,2026-11-01,CH_OTA_BKG,CORP,1,173.75,Checked-Out,True,0.18,31.275,142.475
1,BK_00001,2025-04-15,2025-07-11,CH_OTA_BKG,NET,1,92.51,Confirmed,False,0.18,0.000,92.510
2,BK_00002,2025-08-27,2025-10-31,CH_OTA_BKG,PROMO,3,1058.85,Checked-Out,True,0.18,190.593,868.257
3,BK_00003,2026-03-06,2026-04-04,CH_WHOLE,CORP,1,398.70,Confirmed,True,0.00,0.000,398.700
4,BK_00004,2026-12-16,2027-02-09,CH_DIRECT,CORP,1,170.43,Cancelled,True,0.00,0.000,170.430


In [11]:
# Basic schema and missing-value overview
df.info()

sample_values = df.apply(lambda s: s.dropna().unique()[:3], axis=0)
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True),
    "sample_values": sample_values.apply(lambda v: ", ".join(map(str, v))),
}).sort_values(by="missing", ascending=False)

display(summary)

duplicate_rows = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_rows:,}")

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   booking_id               5000 non-null   str           
 1   booking_date             5000 non-null   datetime64[us]
 2   check_in_date            5000 non-null   datetime64[us]
 3   channel_id               5000 non-null   str           
 4   rate_code_id             5000 non-null   str           
 5   rooms_sold               5000 non-null   int64         
 6   gross_room_revenue       5000 non-null   float64       
 7   status                   5000 non-null   str           
 8   is_commissionable        5000 non-null   bool          
 9   default_commission_rate  5000 non-null   float64       
 10  commission_amount        5000 non-null   float64       
 11  net_room_revenue         5000 non-null   float64       
dtypes: bool(1), datetime64[us](2), float64(4), in

,dtype,missing,missing_pct,n_unique,sample_values
booking_id,str,0,0.0,5000,"BK_00000, BK_00001, BK_00002"
booking_date,datetime64[us],0,0.0,729,"2026-10-17 00:00:00, 2025-04-15 00:00:00, 2025..."
check_in_date,datetime64[us],0,0.0,796,"2026-11-01 00:00:00, 2025-07-11 00:00:00, 2025..."
channel_id,str,0,0.0,4,"CH_OTA_BKG, CH_WHOLE, CH_DIRECT"
rate_code_id,str,0,0.0,4,"CORP, NET, PROMO"
rooms_sold,int64,0,0.0,3,"1, 3, 2"
gross_room_revenue,float64,0,0.0,4900,"173.75, 92.51, 1058.85"
status,str,0,0.0,3,"Checked-Out, Confirmed, Cancelled"
is_commissionable,bool,0,0.0,2,"True, False"
default_commission_rate,float64,0,0.0,3,"0.18, 0.0, 0.15"


Duplicate rows: 0


In [12]:
# Numeric checks: distribution, negatives, and outliers
numeric_cols = df.select_dtypes(include="number").columns

if len(numeric_cols) == 0:
    print("No numeric columns found.")
else:
    display(df[numeric_cols].describe().T)

    negative_counts = (df[numeric_cols] < 0).sum().sort_values(ascending=False)
    display(pd.DataFrame({"negative_count": negative_counts}))

    outlier_rows = []
    for col in numeric_cols:
        series = df[col].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        outlier_rows.append({
            "column": col,
            "outlier_count": int(outliers),
            "outlier_pct": round(outliers / len(series) * 100, 2),
            "min": series.min(),
            "max": series.max(),
        })

    if outlier_rows:
        display(pd.DataFrame(outlier_rows).sort_values(by="outlier_count", ascending=False))

,count,mean,std,min,25%,50%,75%,max
rooms_sold,5000.0,2.000600,0.818495,1.0000,1.000,2.000,3.0000,3.0000
gross_room_revenue,5000.0,578.826806,352.859140,80.0100,299.085,477.315,814.7175,1495.1300
default_commission_rate,5000.0,0.083682,0.083312,0.0000,0.000,0.150,0.1800,0.1800
commission_amount,5000.0,35.672066,58.043867,0.0000,0.000,0.000,58.5240,266.9364
net_room_revenue,5000.0,543.154740,336.803055,68.0085,276.910,451.085,755.8350,1495.1300


,negative_count
rooms_sold,0
gross_room_revenue,0
default_commission_rate,0
commission_amount,0
net_room_revenue,0


,column,outlier_count,outlier_pct,min,max
3,commission_amount,358,7.16,0.0000,266.9364
4,net_room_revenue,26,0.52,68.0085,1495.1300
0,rooms_sold,0,0.00,1.0000,3.0000
2,default_commission_rate,0,0.00,0.0000,0.1800
1,gross_room_revenue,0,0.00,80.0100,1495.1300


In [13]:
# Categorical and date checks
cat_cols = df.select_dtypes(include=["object", "category", "bool", "string"]).columns

if len(cat_cols) == 0:
    print("No categorical columns found.")
else:
    cat_rows = []
    whitespace_rows = []
    for col in cat_cols:
        series = df[col]
        value_counts = series.value_counts(dropna=True)
        top_value = value_counts.index[0] if not value_counts.empty else None
        top_freq = int(value_counts.iloc[0]) if not value_counts.empty else 0
        cat_rows.append({
            "column": col,
            "n_unique": int(series.nunique(dropna=True)),
            "missing": int(series.isna().sum()),
            "top_value": top_value,
            "top_freq": top_freq,
        })

        text = series.dropna().astype(str)
        stripped = text.str.strip()
        whitespace_rows.append({
            "column": col,
            "empty_or_whitespace": int((stripped == "").sum()),
            "leading_trailing_ws": int((stripped != text).sum()),
        })

    display(pd.DataFrame(cat_rows).sort_values(by="n_unique", ascending=False))
    display(pd.DataFrame(whitespace_rows).sort_values(by="leading_trailing_ws", ascending=False))

if date_cols:
    today = pd.Timestamp.today().normalize()
    date_summary = []
    for col in date_cols:
        series = df[col]
        date_summary.append({
            "column": col,
            "min": series.min(),
            "max": series.max(),
            "missing": int(series.isna().sum()),
            "future_dates": int((series > today).sum()),
        })

    display(pd.DataFrame(date_summary))

# Simple consistency checks if common fields exist
checks = {}
if {"booking_date", "check_in_date"}.issubset(df.columns):
    checks["check_in_before_booking"] = int((df["check_in_date"] < df["booking_date"]).sum())
if {"check_in_date", "check_out_date"}.issubset(df.columns):
    checks["check_out_before_check_in"] = int((df["check_out_date"] < df["check_in_date"]).sum())

if checks:
    display(pd.DataFrame([checks]))


,column,n_unique,missing,top_value,top_freq
0,booking_id,5000,0,BK_00000,1
1,channel_id,4,0,CH_OTA_BKG,1287
2,rate_code_id,4,0,PROMO,1295
3,status,3,0,Cancelled,1703
4,is_commissionable,2,0,True,3749


,column,empty_or_whitespace,leading_trailing_ws
0,booking_id,0,0
1,channel_id,0,0
2,rate_code_id,0,0
3,status,0,0
4,is_commissionable,0,0


,column,min,max,missing,future_dates
0,booking_date,2025-01-01,2026-12-31,0,1813
1,check_in_date,2025-01-03,2027-03-28,0,2132


,check_in_before_booking
0,0
